In [11]:
# == Imports ==
import os
import json
import sys
import shutil
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy.signal as signal
from datasets import load_dataset
from IPython.display import Audio, display
from dotenv import load_dotenv
import jiwer
import platform
import time
import warnings
from pprint import pprint
import pickle
import joblib
from pathlib import Path
import gradio as gr
from itertools import islice
import modal


%reload_ext autoreload
%autoreload 2

# Project imports
from asr_pipeline_benchmark.configuration import Config
from asr_pipeline_benchmark.storage import LocalStorage

from asr_pipeline_benchmark.models import load_transcriber_engine, get_model_backend, run_batch_inference

from asr_pipeline_benchmark.preprocessing import (
    cleanup_pipeline, add_white_noise, add_synthetic_reverberation,
    extract_ir_from_mit_dataset, extract_irs_from_mit_dataset_by_location,
    add_real_reverberation, add_smart_audio_dropouts,
    packet_loss, band_limit, hard_clip_ratio, hard_clip_ratio_norm,
    codec_degrade
)
from asr_pipeline_benchmark.utility import (
    generate_sine_tone, generate_run_label, batch_iterator,
    cleanup_results_col_names, expand_mfcc_results
)
from asr_pipeline_benchmark.audio_analysis import analyze_asr_audio_health
from asr_pipeline_benchmark.data import get_mit_ir_dataset, get_speech_dataset
from asr_pipeline_benchmark.visualization import plot_feature_scatter_grid

# Filter out the benign duplicate logits processor warnings from transformers
warnings.filterwarnings("ignore", message=".*logits processor of type.*")

load_dotenv()

# Set figsize global params
plt.rcParams['figure.figsize'] = (12,5)

modalApp = modal.App("asr-benchmark-pipeline")

config = Config()

In [ ]:
# # ==== CONFIGURATION ====
#
# # == PATHS ==
# NOTEBOOK_DIR = Path.cwd()
# TRANSCRIBE_CACHE_DIR = NOTEBOOK_DIR / "results" / "transcribe_cache"
# CHECKPOINT_DIR = TRANSCRIBE_CACHE_DIR / "checkpoint"
# CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
#
# RUN_NOISE_TESTS = False
# LAUNCH_GRADIO = False
#
# # Flag if running on Apple Silicon (M1) to load MLX-WHISPER for MLX models.
# IS_APPLE_SILICON = platform.system() == "Darwin" and platform.processor() == "arm"
#
# # == MODEL FLAGS ==
# # MODEL_SMALL determines which size of transcriber engines is loaded.
# # Based on the model names below
# MODEL_SMALL = True  # Set False to run large models
# MLX_SMALL_MODEL_NAME = "mlx-community/whisper-small-mlx"
# MLX_LARGE_MODEL_NAME = "mlx-community/whisper-large-v3-turbo"
# HF_TRANSFORMER_SMALL_MODEL_NAME = "openai/whisper-small"
# HF_TRANSFORMER_LARGE_MODEL_NAME = "openai/whisper-large-v3-turbo"
#
#
# # == Benchmark Flags ==
# # inference runs will verify first if a previous run has been cached as pickle file in TRANSCRIBE_CACHE_DIR
# # and load it first if available. Otherwise will run inference.
# # Force train runs inference if previous run is present and overwrites the file.
# # runs are labeled as such: <type>__<modelname>__<numberOfSamples>__<postfixInfo>.pkl
# # if any of those variables do not match, the inference run is considered different and will run.
# FORCE_RETRAIN = False
# RUN_BENCHMARK_INFERENCE = True
# IS_MOCK = True  # For future implementation. Not used at the moment.
# BATCH_SIZE = 50
#
# # == Speed Comparison Flags ==
# RUN_FRAMEWORK_SPEED_COMPARISON_TEST = False
# # To run frameworkd comparison between 'mlx' and 'mps', need to force it.
# # Otherwise, will run on 'cpu'
# FORCE_MPS = True


# Load dataset

In [3]:
# == Load dataset in stream mode ==

print(f"Loading LibriSpeech dataset stream...")
# dataset = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True)
dataset = get_speech_dataset()

Loading LibriSpeech dataset stream...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

In [ ]:
# == real room IR reverberation ==
# ir_ds = load_dataset("benjamin-paine/mit-impulse-response-survey", split="train")
ir_ds = get_mit_ir_dataset()

location_id2label = ir_ds.features["location"].int2str
location_str2id = ir_ds.features["location"].str2int
ir_ds_locations = ir_ds.features['location'].names
#pprint(ir_ds_locations)

## Quick tests on data

In [ ]:
# == Grab first sample ==
iterator = iter(dataset)
sample = next(iterator)
# sample2 = next(iterator)
# sample2 = next(iterator)
audio_array = sample["audio"]["array"]
sample_rate = sample["audio"]["sampling_rate"]
ground_truth_text = sample['text']

print(f"Sample Rate: {sample_rate} Hz")
print(f"Ground truth text: {ground_truth_text}")

In [ ]:
sample = next(iterator)
sample

In [ ]:
# == Plot the clean waveform ==
librosa.display.waveshow(audio_array, sr=sample_rate)
plt.title('Clean Speech Waveform - Sample 1')
plt.show()

In [ ]:
# == Listen to the audio ==
Audio(audio_array, rate=sample_rate)

# Metrics and Utility Functions

In [ ]:
# == fct: cleanup_pipeline (jiwer compose) ==
# cleanup_pipeline = jiwer.Compose([
#     jiwer.ToLowerCase(),
#     jiwer.RemovePunctuation(),
#     jiwer.Strip(),
#     # jiwer.ReduceToListOfListOfWords()
# ])

# def normalize_text(text):
#     return jiwer.RemovePunctuation()(text.lower().strip())


In [ ]:
# == fct: analyze_asr_audio_health ==

# def analyze_asr_audio_health(y, frame_length=2048, hop_length=512):
#     """
#     Calculates audio metrics and flags structural issues that degrade ASR
#     Assumes y is a 1D float32 numpy array
#     Returns a dictionary of metrics and ASR flags
#     """
#     eps = 1e-10
#
#     # ----------------------------------------
#     # 1. Base energy and metrics
#     # ----------------------------------------
#     # Global RMS
#     rms_linear_global = np.sqrt(np.mean(y**2))
#     rms_dbfs_global = 20 * np.log10(np.maximum(rms_linear_global, eps))
#
#     # Frame-based RMS
#     rms_frames = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
#     rms_dbfs_frames = 20 * np.log10(np.maximum(rms_frames, eps))
#
#     # Linear and DBFS Peaks
#     abs_y = np.abs(y)
#     peak_max_linear = np.max(abs_y)
#     peak_max_dbfs = 20 * np.log10(np.maximum(peak_max_linear, eps))
#
#     # ----------------------------------------
#     # 2. ASR Specific Diagnostic Metrics
#     # ----------------------------------------
#     # DC Offset: Should be very close to 0.0
#     dc_offset = np.mean(y)
#
#     # Crest Factor: Peak to RMS Ratio in dB
#     crest_factor_db = peak_max_dbfs - rms_dbfs_global
#     #
#     # Zero-Crossing Rate: High ZCR + Low RMS = Pure Noise
#     zcr_frames = librosa.feature.zero_crossing_rate(y=y, frame_length=frame_length, hop_length=hop_length)[0]
#     zcr_mean = np.mean(zcr_frames)
#
#     # ----------------------------------------
#     # 3. Automated ASR Health Flags
#     # ----------------------------------------
#     # Clip Detection: Values Tracking exactly at 1.0 or -1.0
#     is_clipping = np.sum(abs_y >=.99) > (len(y) * 0.001)  # More than 0.1% of sampels
#
#     # Too Quiet Detection: ASR models struggle below -40 dBFS RMS
#     is_too_quiet = rms_dbfs_global < -40.0
#
#     # Pure Noise/Silence Detection: High zero crossing but no real energy
#     # is_pure_noise = (zcr_mean > 0.15) and (rms_dbfs_global < -30.0)
#
#     zcr_min = np.min(zcr_frames)    # Vowels always drop ZCR low, noise dow not
#     zcr_std = np.std(zcr_frames)    # Speech has high ZCR variance; static has low variance
#     # If the minimum frame ZCR is high, it means the signal never became harmonic (no vowels)
#     is_pure_noise = (zcr_min > 0.12) and (rms_dbfs_global < -25.0)
#
#     round_factor = 4
#     return {
#         "metrics": {
#             "rms_lin_global": round(float(rms_linear_global), round_factor),
#             "rms_dbfs_global": round(float(rms_dbfs_global), round_factor),
#             "rms_dbfs_min_frame": round(float(np.min(rms_dbfs_frames)), round_factor),
#             "rms_dbfs_max_frame": round(float(np.max(rms_dbfs_frames)), round_factor),
#             "linear_range": [round(float(np.min(y)), round_factor), round(float(np.max(y)), round_factor)],
#             "peak_dbfs": round(float(peak_max_dbfs), round_factor),
#             "dc_offset": round(float(dc_offset), round_factor),
#             "crest_factor_db": round(float(crest_factor_db), round_factor),
#             "min_zcr": round(float(zcr_min), round_factor),
#             "std_zcr": round(float(zcr_std), round_factor),
#             "mean_zcr": round(float(zcr_mean), round_factor)
#         },
#         "asr_flags": {
#             "action_required": bool(is_clipping or is_too_quiet or is_pure_noise or (abs(dc_offset) > .01)),
#             "reasons": {
#                 "digital_clipping_detected": bool(is_clipping),
#                 "signal_too_faint": bool(is_too_quiet),
#                 "likely_static_or_hum": bool(is_pure_noise),
#                 "dc_offset_warning": bool(abs(dc_offset) > 0.01)
#             }
#         }
#     }

In [ ]:
# == tone generator ==

# def generate_sine_tone(frequency=440.0, duration=2.0, sr=16_000, amplitude=1.0):
#     """
#     Generate a pure sine tone array perfectly compatible with librosa for testing.
#     """
#     # Create the time axis array
#     t = np.linspace(0, duration, int(sr * duration), endpoint=False)
#
#     # Generate the sine wave and scale by amplitude (1.0 is full scale / 0 dBFS peak)
#     y = amplitude * np.sin(2 * np.pi * frequency * t)
#
#     # Ensure it uses float32
#     return y.astype(np.float32), sr

# Audio Modification/Corruption Functions

## White Gaussian Noise

In [ ]:
# # == fct: add_white_noise ==
#
# def add_white_noise(signal, target_snr_db):
#     """
#     Inject additive white Gaussian noise scaled to a precise target SNR (dB)
#     """
#     # Calculate RMS of clean signal
#     rms_signal = np.sqrt(np.mean(np.square(signal)))
#
#     # Calculate the required RMS of noise based on target SNR in dB
#     rms_noise = rms_signal / (10**(target_snr_db/20))
#
#     # Generate noise and scale it
#     noise_raw = np.random.randn(len(signal))
#     rms_noise_raw = np.sqrt(np.mean(np.square(noise_raw)))
#
#     # Scale the noise to match the target noise RMS
#     noise_scaled = noise_raw * (rms_noise / rms_noise_raw)
#
#     signal_with_noise = signal + noise_scaled
#
#     if np.max(np.abs(signal_with_noise))>1.0:
#         print("Warning: Mixed audio exceeds 0 dBFS. Normalizing to prevent clipping.")
#         signal_with_noise = signal_with_noise / np.max(np.abs(signal_with_noise))
#
#     return signal_with_noise


### White Noise tests

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_0dB = add_white_noise(audio_array, 0)
    pprint(analyze_asr_audio_health(test_signal_0dB))
    librosa.display.waveshow(test_signal_0dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({0} dB SNR White Noise)")
    plt.legend()
    plt.show()

    Audio(test_signal_0dB, rate=sample_rate)

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_3dB = add_white_noise(audio_array, 3)
    pprint(analyze_asr_audio_health(test_signal_3dB))
    librosa.display.waveshow(test_signal_3dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({3} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_3dB, rate=sample_rate)


In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_6dB = add_white_noise(audio_array, 6)
    pprint(analyze_asr_audio_health(test_signal_6dB))
    librosa.display.waveshow(test_signal_6dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({6} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_6dB, rate=sample_rate)


In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_9dB = add_white_noise(audio_array, 9)
    pprint(analyze_asr_audio_health(test_signal_9dB))
    librosa.display.waveshow(test_signal_9dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({9} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_9dB, rate=sample_rate)

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_12dB = add_white_noise(audio_array, 12)
    pprint(analyze_asr_audio_health(test_signal_12dB))
    librosa.display.waveshow(test_signal_12dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({12} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_12dB, rate=sample_rate)

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_20dB = add_white_noise(audio_array, 20)
    pprint(analyze_asr_audio_health(test_signal_20dB))
    librosa.display.waveshow(test_signal_20dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({20} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_20dB, rate=sample_rate)

In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_neg3dB = add_white_noise(audio_array, -3)
    pprint(analyze_asr_audio_health(test_signal_neg3dB))
    librosa.display.waveshow(test_signal_neg3dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({-3} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_neg3dB, rate=sample_rate)


In [ ]:
if config.RUN_NOISE_TESTS:
    test_signal_neg6dB = add_white_noise(audio_array, -6)
    pprint(analyze_asr_audio_health(test_signal_neg6dB))
    librosa.display.waveshow(test_signal_neg6dB, sr=sample_rate, alpha=0.5, label='noisy')
    librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
    plt.title(f"Audio Waveform Overlay ({-6} dB SNR White Noise)")
    plt.legend()
    plt.show()
    Audio(test_signal_neg6dB, rate=sample_rate)

## Reverberation

In [ ]:
# == fct: add_synthetic_reverberation() ==

# def add_synthetic_reverberation(signal, sr=16_000, rt60=0.5, mix=0.4):
#     """
#     Synthesizes an exponential decay Impulse Response (IR) to simulate room acoustics,
#     then convolve it with the clean speech signal
#
#     rt6-: Time (seconds) for the reverb tail to drop by 60dB.
#     mix: dry/wet mix balance between direct signal and reverberation
#     """
#     ir_len = int(rt60 * sr)
#     t = np.linspace(0, rt60, ir_len)
#
#     # Generate white noise and apply clean mathematical exponential devay envelope
#     # -6.91 guarantees the amplitude drops exactly 60 dB over the course of rt60
#     decay_envelope = np.exp(-6.91 * t / rt60)
#     ir = np.random.randn(ir_len) * decay_envelope
#
#     # Normalize the IR energy fader to prevent massive volume scaling jumps
#     # In signal processing, we use L2 normalization to keep the total amount of enery = 1
#     ir = ir / np.sqrt(np.sum(ir**2))
#
#     # Convolve the signal with our custom synthetic room IR
#     reverb_signal = np.convolve(signal, ir, mode='full')[:len(signal)]
#
#     # Apply standard Wet/Dry mixing balance
#     mixed_signal = ( 1-mix ) * signal + mix * reverb_signal
#
#     # Safety rail guard to prevent hard clipping
#     if np.max(np.abs(mixed_signal)) > 1.0:
#         mixed_signal = mixed_signal / np.max(np.abs(mixed_signal))
#
#     return mixed_signal

In [ ]:
# == fct: add_real_reverberation() ==

# def extract_ir_from_mit_dataset(dataset, ir_idx, target_sr=16_000):
#     """
#     Retreive the IR audio array fromthe MIT Impulse Respons survey dataset based on its id
#     """
#     sample = dataset.filter(lambda sample: sample["id"]==ir_idx)
#     decoder = sample["audio"][0]
#     audio_tensor = decoder.get_all_samples()
#     y_native = audio_tensor.data.numpy()
#     sr_native = sample["audio"][0].metadata.sample_rate
#
#     y = librosa.resample(y_native, orig_sr=sr_native, target_sr=target_sr)
#
#     return y, target_sr
#
#
# def extract_irs_from_mit_dataset_by_location(dataset, location, target_sr=16_000):
#     """
#     Retreive the IR audio array fromthe MIT Impulse Respons survey dataset based on its id
#     """
#     examples = dataset.filter(lambda sample: sample["location"]==location)
#
#     results = []
#
#     for example in examples:
#         idx = example["id"]
#         detail = example["detail"]
#         decoder = example["audio"]
#         audio_tensor = decoder.get_all_samples()
#         y_native = audio_tensor.data.numpy()
#         sr_native = example["audio"].metadata.sample_rate
#         duration = example["audio"].metadata.duration_seconds
#
#         y = librosa.resample(y_native, orig_sr=sr_native, target_sr=target_sr)
#
#         results.append(
#             {'id': idx,
#              'location': location,
#              'detail': detail,
#              'duration': duration,
#              'audio': y,
#              'sr': target_sr
#              })
#
#     return results
#
#
# def add_real_reverberation(signal, ir, sr=16_000, mix=0.4):
#     """
#     Convolves a clean audio signal with a recorder physical Impulse Response
#     """
#
#     # Force mono on both signal and IR
#     # if len(signal.shape)> 1:
#     #     signal = np.mean(signal, axis=1)
#     # if len(ir.shape) > 1:
#     #     ir = np.mean(ir, axis=0)
#
#     # Flatten ir array if necessary
#     if ir.shape[0]==1:
#         ir = ir.flatten()
#
#
#     # Normalize the IR energy so it doesn't artificially boost the overall volume
#     # using L-infinity normalization to ensure that the high-amplitude transient of the impulse response
#     # does not cause L2 normalization to overly attenuate the decay tail.
#     # ir_scaled = ir / np.max(np.abs(ir))
#     # ir_scaled = ir / np.sqrt(np.sum(ir**2))
#     #
#     rms = np.sqrt(np.mean(ir**2))
#     target_rms = 0.05
#     ir_scaled = ir * (target_rms / rms)
#     ir_scaled = np.clip(ir_scaled, -1.0, 1.0)
#
#     # Convolve the signal with the real room IR
#     reverb_signal = np.convolve(signal, ir_scaled, mode='full')[:len(signal)]
#
#     # Apply the wer/dry balance
#     mixed_signal = ( 1 - mix ) * signal + mix * reverb_signal
#
#
#     # Safety prevention of clipping
#     if np.max(np.abs(mixed_signal)) > 1.0:
#         mixed_signal = mixed_signal / np.max(np.abs(mixed_signal))
#
#     return mixed_signal

### Inspection of different normlization effect on IR RMS and dynamic

In [ ]:
irs = extract_irs_from_mit_dataset_by_location(ir_ds, location_str2id("Stairwell"))
print("".join([f"id: {e['id']} \ndetail: {e['detail']} \nduration in secs: {e['duration']} \n---------\n\n" for e in irs]))

In [ ]:
test_ir = irs[1]['audio']
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(test_ir, color='red', alpha=0.5)

In [ ]:
l_inf = test_ir / np.max(np.abs(test_ir))
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(l_inf, color='red', alpha=0.5)

In [ ]:
l2 = test_ir / np.sqrt(np.sum(test_ir**2))
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(l_inf, color='red', alpha=0.5)

In [ ]:
rms = np.sqrt(np.mean(test_ir**2))
target_rms = 0.03
ir_scaled = test_ir * (target_rms / rms)
ir_scaled = np.clip(ir_scaled, -1.0, 1.0)
librosa.display.waveshow(audio_array, color='blue', alpha=0.5)
librosa.display.waveshow(ir_scaled, color='red', alpha=0.5)

# CORRUPTION EVALUATION

In [ ]:
# ==  GRADIO LAUNCHER - Packet Loss ==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:

        gr.Markdown(f"Packet loss")
        with gr.Column():
            for val in [(40, .10), (30, .20), (20, .30), (10, .50), (5, .70)]:
                audio = packet_loss(audio_array, sr=16_000, packet_ms=val[0], loss_rate=val[1])
                # Convert float32 [-1.0, 1.0] to int16 [-32786, 32767]
                audio_int16 = (audio * 32767).astype(np.int16)
                gr_audio = (16_000, audio_int16)
                output = gr.Audio(gr_audio, label=f"{val[1]*100}% packet loss of length of {val[0]}ms")

    demo.launch(prevent_thread_lock=True)

In [ ]:
# ==  GRADIO LAUNCHER - Smart Audio Dropouts ==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:

        gr.Markdown(f"Packet loss")
        with gr.Column():
            for num in [5, 10, 15, 20]:
                for dur in [50, 75, 100]:
                    audio = add_smart_audio_dropouts(audio_array, sr=16_000, num_dropouts=num,
                                                     dropout_duration_ms=dur, threshold_below_global_db=15)

                    audio_int16 = (audio * 32767).astype(np.int16)
                    gr_audio = (16_000, audio_int16)
                    output = gr.Audio(gr_audio, label=f"{num} dropouts of {dur}ms each, threshold below {15}dB")

    demo.launch(prevent_thread_lock=True)

In [ ]:
# ==  GRADIO LAUNCHER -  BAND LIMITING ==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:

        gr.Markdown(f"Packet loss")
        with gr.Column():
            for lc, hc in [(800, 3000), (600, 2000), (500, 1000)]:
                audio = band_limit(audio_array, sr=16_000, lowcut=lc, highcut=hc)
                # Convert float32 [-1.0, 1.0] to int16 [-32786, 32767]
                audio_int16 = (audio * 32767).astype(np.int16)
                gr_audio = (16_000, audio_int16)
                output = gr.Audio(gr_audio, label=f"Band limiting on low cut at {lc} and high cut at {hc}")

    demo.launch(prevent_thread_lock=True)

In [ ]:
irs = extract_irs_from_mit_dataset_by_location(ir_ds, location_str2id("Stairwell"))
print("".join([f"id: {e['id']} \ndetail: {e['detail']} \nduration in secs: {e['duration']} \n---------\n\n" for e in irs]))

In [ ]:
# ==  GRADIO LAUNCHER - REAL REVERB ==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:
        n = len(irs)
        gr.Markdown(f"# {n} different IRs applied to audio")
        with gr.Column():
            for ir in irs:
                audio = add_real_reverberation(audio_array, ir["audio"])
                audio_int16 = (audio * 32767).astype(np.int16)
                gr_audio = (16_000, audio_int16)
                output = gr.Audio(gr_audio, label=f"ID:{ir["id"]} | {location_id2label(ir['location'])}: {ir['detail']} | {round(ir['duration'],2)} secs")

    demo.launch(prevent_thread_lock=True)

In [ ]:
# ==  GRADIO LAUNCHER - HARD AND SOFT CLIPPING==

# safety close of potentially previously running instance of gradio
gr.close_all()

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:
        gr.Markdown(f"# Hard clip and normzalized hard clip applied to audio on different thresholds")
        with gr.Column():
            for th in [.5, .3, .2, .1, .05, .025]:
                audio1 = hard_clip_ratio(audio_array, threshold=th)
                audio1_int16 = (audio1 * 32767).astype(np.int16)
                gr_audio1 = (16_000, audio1_int16)
                output = gr.Audio(gr_audio1, label=f"Hard clip at {th} threshold")

                audio2 = hard_clip_ratio_norm(audio_array, threshold=th)
                audio2_int16 = (audio2 * 32767).astype(np.int16)
                gr_audio2 = (16_000, audio2_int16)
                output = gr.Audio(gr_audio2, label=f"Hard clip normalized at {th} threshold")

    demo.launch(prevent_thread_lock=True)

In [ ]:
# === GRADIO LAUCHER - Bitrate / Codec degradation

gr.close_all()

# Each codec has its own legal bitrate range — Vorbis can't go below ~45k.
BITRATE_MATRIX = {
    "libopus":    ["6k", "8k"]
}

if config.LAUNCH_GRADIO:
    with gr.Blocks() as demo:
        gr.Markdown(f"# Bitrate / Codec degradation")
        with gr.Column():
            for codec, bitrates in BITRATE_MATRIX.items():
                for bitrate in bitrates:
                    try:
                        audio = codec_degrade(audio_array, sr=16_000, bitrate=bitrate, codec=codec)
                    except (ValueError, RuntimeError) as e:
                        # Don't let one unencodable pair tear down the whole demo.
                        print(f"Skipping {codec} @ {bitrate}: {e}")
                        continue

                    audio_int16 = (audio * 32767).astype(np.int16)
                    gr_audio = (16_000, audio_int16)
                    output = gr.Audio(gr_audio, label=f"{codec} at {bitrate} bitrate")

    demo.launch(prevent_thread_lock=True)

The target RMS normalization with clipping worked the best. This is the normalization method implemented in the add_real_reverberation function

In [ ]:
gr.close_all()

# Transcriber Engine

In [ ]:
# == setup classifier pieline ==
# def load_transcriber_engine(IS_APPLE_SILICON=False, FORCE_MPS=False):
#     """
#     Automatically detect if Apple Silicon and tries to load mlx_whisper if available
#     If not available, falls back to HuggingFace pipeline and setup device accordingly ("cuda", "mps", "cpu")
#
#     Needs GLOBAL VAR: IS_APPLE_SILICON  to be set True or False
#     :return:
#         if MLX is used returns the mlx transcriber, otherwise
#     """
#     loaded_model_name = ""
#     if IS_APPLE_SILICON:
#         try:
#             # from mlx_whisper.load_models import load_model as mlx_load_model
#             from mlx_whisper import transcribe as mlx_transcribe
#             print("Apple Silicon Detected. Initializing MLX.")
#
#             def mlx_transcriber(audio_array):
#                 # Using MLX community pre-converted weights of the turbo model
#                 if MODEL_SMALL:
#                     result = mlx_transcribe(audio_array, path_or_hf_repo=MLX_SMALL_MODEL_NAME)
#                     loaded_model_name = MLX_SMALL_MODEL_NAME
#                     # model = mlx_load_model(MLX_SMALL_MODEL_NAME)
#                 else:
#                     result = mlx_transcribe(audio_array, path_or_hf_repo=MLX_LARGE_MODEL_NAME)
#                     loaded_model_name = MLX_LARGE_MODEL_NAME
#                     # model = mlx_load_model(MLX_LARGE_MODEL_NAME)
#
#                 # print(f"Transcriber engine initialized. ({loaded_model_name})")
#                 return {"text": result["text"]}
#             return mlx_transcriber
#
#         except ImportError:
#             print(f"Apple Silicon detected but 'mlx_whisper' not installed. Falling back to PyTorch.")
#
#     # 2 Standard Production/Linux Fallback
#     from transformers import pipeline, AutoTokenizer, GenerationConfig
#     import torch
#
#     device = "cuda" if torch.cuda.is_available() else ("mps" if (IS_APPLE_SILICON or FORCE_MPS) else "cpu")
#     print(f"Standard architecture detected. Initializing Hugging Face pipeline on device: {device}")
#     dtype = torch.float32 if device=="mps" else torch.float16
#
#     if MODEL_SMALL:
#         model_id = HF_TRANSFORMER_SMALL_MODEL_NAME
#     else:
#         model_id = HF_TRANSFORMER_LARGE_MODEL_NAME
#     tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)
#
#     whisper_config = GenerationConfig.from_pretrained(model_id)
#     whisper_config.task = "transcribe"
#     whisper_config.language = "en"
#     # whisper_config.num_beams = 1
#
#     whisper_pipe = pipeline(
#         "automatic-speech-recognition",
#         model=model_id,
#         tokenizer=tokenizer,
#         dtype=dtype,
#         device=device
#     )
#
#     # Enclose pipeline in a wrapper function to pass the manual config along
#     def hf_transcriber(audio_array):
#         result = whisper_pipe(audio_array, generation_config=whisper_config)
#         # print(f"Transcriber engine initialized. ({model_id})")
#         return result
#
#     return hf_transcriber

In [ ]:
# == GENERATE RUN LABEL ==

# def generate_run_label(test_type, model_backend, n_samples, MODEL_SMALL=True, postfix=""):
#     if model_backend =="mlx":
#         if MODEL_SMALL:
#             model_name = config.MLX_SMALL_MODEL_NAME.replace("/", "_")
#         else:
#             model_name = config.MLX_LARGE_MODEL_NAME.replace("/", "_")
#     else:
#         if MODEL_SMALL:
#             model_name = config.HF_TRANSFORMER_SMALL_MODEL_NAME.replace("/", "_")
#         else:
#             model_name = config.HF_TRANSFORMER_LARGE_MODEL_NAME.replace("/", "_")
#
#     if len(postfix) > 0:
#         return f"{test_type}__{model_name}__{n_samples}samples__{postfix}"
#     else:
#         return f"{test_type}__{model_name}__{n_samples}samples"


# def get_model_backend(model) -> str:
#     """Returns 'mlx' or 'torch' based on the instantiated model object"""
#     module_name = getattr(model, "__name__", "")
#
#     if "mlx" in module_name:
#         return "mlx"
#     elif "torch" in module_name or "transformers" or"hf_" in module_name:
#         return "torch"
#     else:
#         return f"unknown ({module_name})"

In [ ]:
# == BATCH ITERATOR ==

# def batch_iterator(iterable, batch_size=50):
#     """Yields small lists of samples (micro-batches) from a streaming iterator."""
#     batch = []
#     for item in iterable:
#         batch.append(item)
#         if len(batch) == batch_size:
#             yield batch
#             batch = []
#     if batch:  # Yield any remaining tail items
#         yield batch

# INFERENCE TESTS

In [ ]:
# == Insantiate transcriber ==
# IS_APPLE_SILICON=True
# transcriber = load_transcriber_engine(config.IS_APPLE_SILICON, model_small=True)

In [ ]:
# == fct: run_benchmark_test ==

# def run_benchmark_test(batch, n_samples=10, corruption_type=None):   # no noise
#     print(f" === RUN BENCHMARK TEST started ===")
#     if corruption_type is None:
#         corruption_type = ["none"]
#     # iterator = iter(dataset)
#     target_snr = 3  # signal-to-noise ratio in dB to be added to the samples
#
#     counter, running_avg_wer, running_avg_cer = 0, 0, 0
#     tot_w_ref, tot_w_hits, tot_w_subs, tot_w_dels, tot_w_ins = 0,0,0,0,0
#     tot_c_ref, tot_c_hits, tot_c_subs, tot_c_dels, tot_c_ins = 0,0,0,0,0
#
#     results = []
#
#     for idx, sample in enumerate(batch):
#         # sample = next(iterator)
#         audio_array = sample["audio"]["array"]
#         sample_rate = sample["audio"]["sampling_rate"]
#
#         for corr_type in corruption_type:
#
#             counter+=1
#             # --------------------------------------
#             # Corruption / Degradation Application
#             # --------------------------------------
#             if corr_type is None:
#                 corr_audio_array = audio_array
#             elif corr_type=="none":
#                 corr_audio_array = audio_array
#             elif corr_type=="whitenoise": # implement the default white noise if no corruption_type parameter is passed
#                 corr_audio_array = add_white_noise(audio_array, target_snr)
#             elif corr_type=="synth-verb":
#                 corr_audio_array = add_synthetic_reverberation(audio_array, mix=0.5)
#             elif corr_type=="real-verb":
#                 ir, sr = extract_ir_from_mit_dataset(ir_ds, 251)
#                 corr_audio_array =  add_real_reverberation(audio_array, ir, mix=0.5)
#             elif corr_type=="dropout":
#                 corr_audio_array = add_smart_audio_dropouts(audio_array)
#             elif corr_type==["whitenoise", "synth-verb", "dropout"]:
#                 corr_audio_array = add_white_noise(audio_array, target_snr)
#                 corr_audio_array = add_synthetic_reverberation(corr_audio_array, mix=0.5)
#                 corr_audio_array = add_smart_audio_dropouts(corr_audio_array)
#             else:
#                 corr_audio_array = audio_array
#
#
#             ref = cleanup_pipeline(sample["text"])
#             hyp = cleanup_pipeline(transcriber(corr_audio_array)['text'])
#
#             # -----------------------------------------------------------------------------
#             # Word Level metrics - WER
#             # -----------------------------------------------------------------------------
#             wer_metrics = jiwer.process_words(ref, hyp)
#             wer = wer_metrics.wer
#             w_hits = wer_metrics.hits
#             w_subs = wer_metrics.substitutions
#             w_dels = wer_metrics.deletions
#             w_ins = wer_metrics.insertions
#             num_words = w_hits + w_subs + w_dels
#
#             # Add to total for macro-averages
#             running_avg_wer = (counter-1) / counter*running_avg_wer + wer/counter
#             tot_w_ref += num_words
#             tot_w_hits += w_hits
#             tot_w_subs += w_subs
#             tot_w_dels += w_dels
#             tot_w_ins += w_ins
#
#
#             # -----------------------------------------------------------------------------
#             # Characeter Level metrics - CER
#             # -----------------------------------------------------------------------------
#             char_metrics = jiwer.process_characters(ref, hyp)
#             cer = char_metrics.cer
#             c_hits = char_metrics.hits
#             c_subs = char_metrics.substitutions
#             c_dels = char_metrics.deletions
#             c_ins = char_metrics.insertions
#             num_chars = c_hits + c_subs + c_dels
#
#             # Add to total for macro-averages
#             running_avg_cer = (counter-1) / counter*running_avg_cer + cer/counter
#             tot_c_ref += num_chars
#             tot_c_hits += c_hits
#             tot_c_subs += c_subs
#             tot_c_dels += c_dels
#             tot_c_ins += c_ins
#
#             # -----------------------------------------------------------------------------
#             # Store results and references into list for further analysis
#             # -----------------------------------------------------------------------------
#             sample_analysis = analyze_asr_audio_health(corr_audio_array)
#
#             results.append({"idx": sample['id'],
#                             "corr_type": corr_type,
#                             "wer": wer,
#                             "w_hits": w_hits,
#                             "w_subs": w_subs,
#                             "w_dels": w_dels,
#                             "w_ins": w_ins,
#                             "num_words": num_words,
#                             "cer": cer,
#                             "c_hits": c_hits,
#                             "c_subs": c_subs,
#                             "c_dels": c_dels,
#                             "c_ins": c_ins,
#                             "num_chars": num_chars,
#                             "sample_analysis": sample_analysis,
#                             "ref": ref,
#                             "hyp": hyp})
#
#
#
#     print(f" === RUN BENCHMARK TEST finished ===")
#     return results



## Run Benchmark Tests

In [ ]:
# == fct: run_individual_inference_on_one_corruption_type ==
# For large ingerence runs

# def run_individual_inference_on_one_corruption_type_old(bench_label, n, corruption_type="whitenoise"):
#     bench_filepath = Path(TRANSCRIBE_CACHE_DIR) / bench_label
#     results = {}
#     cache_loaded_successfully = False
#
#     if bench_filepath.exists() and not FORCE_RETRAIN:
#         try:
#             with open(bench_filepath, "rb") as f:
#                 results = pickle.load(f)
#             cache_loaded_successfully = True
#             print(f"Loaded test results from cache: {bench_filepath}")
#
#         except (pickle.UnpicklingError, IOError, EOFError, AttributeError) as e:
#             print(f"Warning: Cache file found but could not be loaded due to error: {e}")
#             print("Falling back to running the benchmark inference...")
#
#     if not cache_loaded_successfully:
#         results = run_benchmark_test(dataset, n_samples=n, corruption_type=corruption_type)
#
#         print(f"Benchmark Inference Tests completed with {n} samples")
#
#         with open(bench_filepath, "wb") as f:
#             pickle.dump(results, f)
#             print(f"Successfully dumped test results to cache: {bench_filepath}")
#
#     return results


# NEED TO RENAME THAT FUNCITON TO SOMETHING LIKE: RUN_BATH_INFERENCE
# def run_batch_inference(iterator, run_label, n, corruption_type=None, clear_cache=False):
#     print(f"===== Run Batch Inference on {n} samples =====")
#     print(f"run_label: {run_label}")
#     print(f"corruption_type: {corruption_type}")
#
#     if corruption_type is None:
#         corruption_type = ["none"]
#
#     run_filepath = config.TRANSCRIBE_CACHE_DIR / f"{run_label}.pkl"
#     results = []
#     cache_loaded_successfully = False
#
#     if run_filepath.exists() and not config.FORCE_RETRAIN:
#         try:
#             with open(run_filepath, "rb") as f:
#                 results = pickle.load(f)
#             cache_loaded_successfully = True
#             print(f"Loaded test results from cache: \n\t{run_filepath}")
#
#         except (pickle.UnpicklingError, IOError, EOFError, AttributeError) as e:
#             print(f"Warning: Cache file found but could not be loaded due to error: {e}")
#             print("Falling back to running the benchmark inference...")
#
#     if not cache_loaded_successfully and config.RUN_BENCHMARK_INFERENCE:
#
#         sample_counter = 0
#         batch_idx = 0
#         limited_iterator = islice(iterator, n)
#
#         existing_batchs = list(config.CHECKPOINT_DIR.glob(run_label+"*batch_*.pkl"))
#         if existing_batchs:
#             batch_indices = [int(f.stem.split("_")[-1]) for f in existing_batchs]
#             batch_idx = max(batch_indices, default=0) + 1
#             sample_counter = batch_idx * config.BATCH_SIZE
#             print(f"Resuming pipeline from batch {batch_idx:04d} (approx sample {sample_counter})...")
#
#             # fast-forward the lazy iterator
#             for _ in range(sample_counter):
#                 next(limited_iterator, None)
#
#         for batch in batch_iterator(limited_iterator, batch_size=config.BATCH_SIZE):
#             print(f"\n=== Batch #{batch_idx} ===")
#             batch_filepath = config.CHECKPOINT_DIR / f"{run_label}_batch_{batch_idx:04d}.pkl"
#
#             batch_results = run_benchmark_test(batch, n_samples=n, corruption_type=corruption_type)
#
#             with open(batch_filepath, "wb") as f:
#                 pickle.dump(batch_results, f)
#                 print(f"Successfully dumped batch {batch_idx} results to cache: \n\t{batch_filepath}")
#
#             batch_idx += 1
#             sample_counter += len(batch)
#
#         # === BUILD ONE FINAL DATA FILE ===
#         batch_files = list(config.CHECKPOINT_DIR.glob(run_label+"*batch_*.pkl"))
#         print(f"len batch_files: {len(batch_files)}")
#
#         # Load and concatenate
#         for file in batch_files:
#             with open(file, "rb") as f:
#                 batch_ = pickle.load(f)
#             results.extend(batch_)
#
#
#         print(f"Benchmark Inference Tests completed with {n} samples")
#
#         with open(run_filepath, "wb") as f:
#             pickle.dump(results, f)
#             print(f"Successfully dumped test results to cache: \n\t{run_filepath}")
#
#         if clear_cache and run_filepath.exists():
#             print(f"Removing cache file: {run_filepath}")
#             for file in batch_files:
#                 file.unlink()
#                 print(f"Removed batch file: {file}")
#
#     return results


In [28]:
# USE THIS AS A CORRUPTION LIST METHOD TO KEEP TRACK OF PARAMETERS USED
ir, _ = extract_ir_from_mit_dataset(ir_idx=251)

corruptions = {
    "none": lambda x: x,
    "WN_3snr": lambda x: add_white_noise(x, 3),
    "SV_0.5": lambda x: add_synthetic_reverberation(x, mix=0.5),
    "RV_0.5_251": lambda x: add_real_reverberation(x, ir, mix=0.5),
    "SDO_8_100_15": lambda x: add_smart_audio_dropouts(x, num_dropouts=8, dropout_duration_ms=100, threshold_below_global_db=15),
    "WN_3-SV_.5-SDO_8_100_15": lambda x: add_smart_audio_dropouts(
                                                add_synthetic_reverberation(
                                                    add_white_noise(x, 3),
                                                mix=0.5),
                                          num_dropouts=8, dropout_duration_ms=100, threshold_below_global_db=15),
    "codec_8k": lambda x: codec_degrade(x, bitrate="8k"),
    "codec_6k": lambda x: codec_degrade(x, bitrate="6k"),
    "PL_40_.1": lambda x: packet_loss(x, packet_ms=40 , loss_rate=0.10),
    "PL_30_.2": lambda x: packet_loss(x, packet_ms=30 , loss_rate=0.20),
    "PL_20_.3": lambda x: packet_loss(x, packet_ms=20 , loss_rate=0.30),
    "PL_10_.4": lambda x: packet_loss(x, packet_ms=10 , loss_rate=0.40),
    "PL_5_.5": lambda x: packet_loss(x, packet_ms=5 , loss_rate=0.50),
    "BL_800_3000": lambda x: band_limit(x, lowcut=800, highcut=3000),
    "BL_600_2000": lambda x: band_limit(x, lowcut=600, highcut=2000),
    "BL_500_1000": lambda x: band_limit(x, lowcut=500, highcut=1000),
    "clip_1": lambda x: hard_clip_ratio(x, 0.1),
    "clip_05": lambda x: hard_clip_ratio(x, 0.05),
    "clip_025": lambda x: hard_clip_ratio(x, 0.025),
    "clipN_1": lambda x: hard_clip_ratio_norm(x, 0.1),
    "clipN_05": lambda x: hard_clip_ratio_norm(x, 0.05),
    "clipN_025": lambda x: hard_clip_ratio_norm(x, 0.025),
}

In [14]:
# # === TEMPORARY SMALL CORRUPTIONS LIST FOR TESTING ===
# corruptions = {
#     "none": lambda x: x,
#     "WN_3snr": lambda x: add_white_noise(x, 3),
#     "SV_0.5": lambda x: add_synthetic_reverberation(x, mix=0.5),
# }

In [29]:
# == RUN DRESS REHEARSAL BENCHMARK TEST ON 10 SAMPELS ==
storageInterface = LocalStorage()
for run_small_model in [True, False]:
    transcriber = load_transcriber_engine(config.IS_APPLE_SILICON, model_small=run_small_model)
    testtype = "dr"
    n = 10
    run_label = generate_run_label(testtype, get_model_backend(transcriber), n, MODEL_SMALL=run_small_model)
    # corruption_type_list = ["none", "whitenoise", "synth-verb", "real-verb", "dropout", ["whitenoise", "synth-verb", "dropout"]]
    # corruption_type_list = ["none", "whitenoise", "synth-verb", "dropout"]
    iterator = iter(dataset)
    dress_test = run_batch_inference(transcriber, iterator, run_label, n, storageInterface, corruption_type=corruptions, clear_cache=True)


Apple Silicon Detected. Initializing MLX.
===== Run Batch Inference on 10 samples =====
run_label: dr__mlx-community_whisper-small-mlx__10samples

=== Batch #0 ===
 === RUN BENCHMARK TEST started ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

 === RUN BENCHMARK TEST finished ===
Successfully dumped batch 0 results to cache: 
	/Users/jobicari/PycharmProjects/noisy-speech-asr-benchmarking-pipeline/results/transcribe_cache/checkpoint/dr__mlx-community_whisper-small-mlx__10samples_batch_0000.pkl

=== Batch #1 ===
 === RUN BENCHMARK TEST started ===
 === RUN BENCHMARK TEST finished ===
Successfully dumped batch 1 results to cache: 
	/Users/jobicari/PycharmProjects/noisy-speech-asr-benchmarking-pipeline/results/transcribe_cache/checkpoint/dr__mlx-community_whisper-small-mlx__10samples_batch_0001.pkl

=== Batch #2 ===
 === RUN BENCHMARK TEST started ===
 === RUN BENCHMARK TEST finished ===
Successfully dumped batch 2 results to cache: 
	/Users/jobicari/PycharmProjects/noisy-speech-asr-benchmarking-pipeline/results/transcribe_cache/checkpoint/dr__mlx-community_whisper-small-mlx__10samples_batch_0002.pkl

=== Batch #3 ===
 === RUN BENCHMARK TEST started ===
 === RUN BENCHMARK TEST finished ===
Successfully dumped batch 3 results to 

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

 === RUN BENCHMARK TEST finished ===
Successfully dumped batch 0 results to cache: 
	/Users/jobicari/PycharmProjects/noisy-speech-asr-benchmarking-pipeline/results/transcribe_cache/checkpoint/dr__mlx-community_whisper-large-v3-turbo__10samples_batch_0000.pkl

=== Batch #1 ===
 === RUN BENCHMARK TEST started ===
 === RUN BENCHMARK TEST finished ===
Successfully dumped batch 1 results to cache: 
	/Users/jobicari/PycharmProjects/noisy-speech-asr-benchmarking-pipeline/results/transcribe_cache/checkpoint/dr__mlx-community_whisper-large-v3-turbo__10samples_batch_0001.pkl

=== Batch #2 ===
 === RUN BENCHMARK TEST started ===
 === RUN BENCHMARK TEST finished ===
Successfully dumped batch 2 results to cache: 
	/Users/jobicari/PycharmProjects/noisy-speech-asr-benchmarking-pipeline/results/transcribe_cache/checkpoint/dr__mlx-community_whisper-large-v3-turbo__10samples_batch_0002.pkl

=== Batch #3 ===
 === RUN BENCHMARK TEST started ===
 === RUN BENCHMARK TEST finished ===
Successfully dumped batc

In [26]:
dress_test

[{'idx': '6930-75918-0000',
  'corr_type': 'none',
  'wer': 0.0,
  'w_hits': 8,
  'w_subs': 0,
  'w_dels': 0,
  'w_ins': 0,
  'num_words': 8,
  'cer': 0.0,
  'c_hits': 46,
  'c_subs': 0,
  'c_dels': 0,
  'c_ins': 0,
  'num_chars': 46,
  'sample_analysis': {'metrics': {'rms_lin_global': 0.0294,
    'rms_dbfs_global': -30.6444,
    'rms_dbfs_min_frame': -61.8058,
    'rms_dbfs_max_frame': -23.6029,
    'linear_range': [-0.2902, 0.2242],
    'peak_dbfs': -10.7472,
    'dc_offset': -0.0,
    'crest_factor_db': 19.8971,
    'min_zcr': 0.0903,
    'std_zcr': 0.1103,
    'mean_zcr': 0.2589,
    'mfccs_mean': [-305.6367,
     14.4794,
     -13.7238,
     57.837,
     -30.6838,
     28.0411,
     -12.8654,
     -4.9441,
     2.267,
     7.8182,
     1.1965,
     -1.0982,
     3.4739],
    'mfccs_std': [102.4731,
     60.858,
     32.2848,
     25.3335,
     19.481,
     17.2421,
     16.9738,
     17.4905,
     11.7579,
     15.369,
     7.7018,
     10.7282,
     12.0947],
    'sf_mean': np.fl

In [ ]:
# === Verification on sample used in both loops ===
dr_run_smallModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "dr__mlx-community_whisper-small-mlx__4samples.pkl", "rb") as f:
    dr_run_smallModel = pickle.load(f)

dr_run_largeModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "dr__mlx-community_whisper-large-v3-turbo__4samples.pkl", "rb") as f:
    dr_run_largeModel = pickle.load(f)

# verify that the samples used are the same for the small and large model transcription benchmark run
dr_small = pd.DataFrame(dr_run_smallModel)
dr_large = pd.DataFrame(dr_run_largeModel)

assert dr_small["idx"].equals(dr_large["idx"]), True
assert dr_small["ref"].equals(dr_large["ref"]), True

# pd.concat([dr_small["idx"], dr_large["idx"], dr_small["ref"], dr_large["ref"]], axis=1)

In [23]:
dr_small

""


In [ ]:
# == LARGE INGERENCE BENCHMARK TEST ==
for run_small_model in [True, False]:
    transcriber = load_transcriber_engine(config.IS_APPLE_SILICON, model_small=run_small_model)
    testtype = "benchtest"
    n=1000
    run_label = generate_run_label(testtype, get_model_backend(transcriber), n, MODEL_SMALL=run_small_model)
    iterator = iter(dataset)
    bench_run = run_batch_inference(transcriber, iterator, run_label, n, corruption_type=corruptions, clear_cache=True)


In [ ]:
# === Verification on sample used in both loops ===
bench_run_smallModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "dr__mlx-community_whisper-small-mlx__8samples.pkl", "rb") as f:
    bench_run_smallModel = pickle.load(f)

bench_run_largeModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "dr__mlx-community_whisper-large-v3-turbo__8samples.pkl", "rb") as f:
    bench_run_largeModel = pickle.load(f)

# verify that the samples used are the same for the small and large model transcription benchmark run
df_small = pd.DataFrame(bench_run_smallModel)
df_large = pd.DataFrame(bench_run_largeModel)

assert df_small["idx"].equals(df_large["idx"]), True
assert df_small["ref"].equals(df_large["ref"]), True

# pd.concat([df_small["idx"], df_large["idx"], df_small["ref"], df_large["ref"]], axis=1)

## Load Large Whisper Model Inference Transcription Results

In [ ]:
bench_run_largeModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "benchtest__mlx-community_whisper-large-v3-turbo__1000samples.pkl", "rb") as f:
    bench_run_largeModel = pickle.load(f)

## Load Small Whisper Model Inference Transcription Results

In [ ]:
bench_run_smallModel = None
with open(config.TRANSCRIBE_CACHE_DIR / "benchtest__mlx-community_whisper-small-mlx__1000samples.pkl", "rb") as f:
    bench_run_smallModel = pickle.load(f)

## EDA and Data Engineering

### Legacy - to be removed

In [ ]:
bench_run = None
with open(config.TRANSCRIBE_CACHE_DIR / "_legacy" / "benchtest__mlx-community_whisper-large-v3-turbo__10samples___FULL.pkl", "rb") as f:
    bench_run = pickle.load(f)

In [ ]:
for idx, data in enumerate(bench_run.values()):
    if idx==0:
        print(pd.json_normalize(data))

In [ ]:
# Normalize each dataset into a list of dataframes
df_list = [pd.json_normalize(data) for data in bench_run.values()]

# Combine them vertically and use the dictionary keys as a new columns to id them
df_tests = pd.concat(df_list, keys=bench_run.keys())

df_tests

In [ ]:
# cols = [col.split(".") for col in df_tests.columns]
#
# exclude = {"metrics", "asr_flags", "reasons"}
#
# cleaned_cols = [
#     ".".join(
#         item.replace("c_analysis", "c_anly")
#         .replace("n_analysis", "n_anly")
#         .replace("digital_clipping_detected", "clipping_detected")
#         for item in sublist
#         if item not in exclude
#     )
#     for sublist in cols
# ]
#
# df_tests.columns = cleaned_cols
# df_tests.info()
# # pprint(cleaned_cols)

In [ ]:
# == Some data engineering ==
# df_tests["delta_wer"] = df_tests["n_wer"] - df_tests["c_wer"]
# df_tests["delta_crest_factor"] = df_tests["n_anly.crest_factor_db"] - df_tests["c_anly.crest_factor_db"]
# df_tests["delta_mean_zcr"] = df_tests["n_anly.mean_zcr"] - df_tests["c_anly.mean_zcr"]
# df_tests["delta_std_zcr"] = df_tests["n_anly.std_zcr"] - df_tests["c_anly.std_zcr"]
# df_tests["delta_min_zcr"] = df_tests["n_anly.min_zcr"] - df_tests["c_anly.min_zcr"]
# df_tests["delta_rms"] = df_tests["n_anly.rms_dbfs_global"] - df_tests["c_anly.rms_dbfs_global"]
# df_tests.sort_values(by=["c_wer", "n_wer"], ascending=False, inplace=True)
#
# df_tests

### New version

In [ ]:
# verify that the samples used ar ethe same for the small and large model transcription benchmark run
df_small = pd.DataFrame(bench_run_smallModel)
df_large = pd.DataFrame(bench_run_largeModel)

df_small["ref"].equals(df_large["ref"])

pd.concat([df_small["idx"], df_large["idx"], df_small["ref"], df_large["ref"]], axis=1)


In [ ]:
df_small_sorted = df_small.sort_values(by=['idx', 'corr_type'], ascending=True)
df_large_sorted = df_large.sort_values(by=['idx', 'corr_type'], ascending=True)

In [ ]:
pd.concat([df_small_sorted["idx"], df_large_sorted["idx"], df_small_sorted["ref"], df_large_sorted["ref"]], axis=1)

In [ ]:
df_smallModel = cleanup_results_col_names(bench_run_smallModel)
df_largeModel = cleanup_results_col_names(bench_run_largeModel)
# df_smallModel.info()
# df_largeModel.info()

In [ ]:
# == Save DF to JSON file
# df_tests.to_json("./results/bench_test_history.json", orient="records", lines=True)

# Read it back into Pandas DF
# loaded_df = pd.read_json("./results/bench_test_history.json", orient="records", lines=True)
# loaded_df.columns

### Flatten MFCCs

In [ ]:
# exp_df = pd.DataFrame(df_smallModel["mtr.mfccs_mean"].tolist(), index=df_smallModel.index)
# exp_df.columns = [f"mtr.mfcc{i}_mean" for i in range(1,14) ]+[f"mtr.mfcc{i}_std" for i in range(1,14) ]
# exp_df

In [ ]:
# === CLeanup DF ===

# Expland MFCCs columns
df_smallModel_exp = expand_mfcc_results(df_smallModel)
df_largeModel_exp = expand_mfcc_results(df_largeModel)

# Convert idx to integers and change col name
df_smallModel_exp["idx"] = pd.factorize(df_smallModel_exp["idx"])[0]
df_smallModel_exp.rename(columns={"idx": "sample_id"})
df_largeModel_exp["idx"] = pd.factorize(df_largeModel_exp["idx"])[0]
df_largeModel_exp.rename(columns={"idx": "sample_id"})

In [ ]:
df_smallModel_exp.info()

In [ ]:
# == generate column lists per type
num_cols = df_smallModel_exp.select_dtypes(include=['number']).columns.tolist()

cat_cols = df_smallModel_exp.select_dtypes(include=['object']).columns.tolist()

print(f"numerical columns: \n\t{num_cols}")
print(f"categorical columns: \n\t{cat_cols}")

In [ ]:
# == Look at cross-correlation between numerical variables ==
cols_to_correlate = [
    'wer',
    'cer',
    'mtr.crest_factor_db',
    'mtr.min_zcr',
    'mtr.std_zcr',
    'mtr.mean_zcr',
    'mtr.rms_lin_global',
    'mtr.rms_dbfs_global',
    'mtr.mfcc1_mean', 'mtr.mfcc2_mean', 'mtr.mfcc3_mean', 'mtr.mfcc4_mean', 'mtr.mfcc5_mean', 'mtr.mfcc6_mean', 'mtr.mfcc7_mean',
    'mtr.mfcc8_mean', 'mtr.mfcc9_mean', 'mtr.mfcc10_mean', 'mtr.mfcc11_mean', 'mtr.mfcc12_mean', 'mtr.mfcc13_mean',
    'mtr.mfcc1_std', 'mtr.mfcc2_std', 'mtr.mfcc3_std', 'mtr.mfcc4_std', 'mtr.mfcc5_std', 'mtr.mfcc6_std', 'mtr.mfcc7_std',
    'mtr.mfcc8_std', 'mtr.mfcc9_std', 'mtr.mfcc10_std', 'mtr.mfcc11_std', 'mtr.mfcc12_std', 'mtr.mfcc13_std',
    'mtr.sf_mean', 'mtr.sf_std'
]

# df_smallModel_exp[cols_to_correlate].info()

corr_df = df_smallModel_exp[cols_to_correlate].dropna()
corr_matrix = corr_df.corr(method='spearman')

plt.figure(figsize=(22, 18))
mask = np.tril(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix,
            mask=mask,
            annot=True,
            fmt='.2f',
            annot_kws={"size":8},
            cmap='coolwarm',
            vmin=-1, vmax=1,
            square=True,
            linewidths=.5,
            cbar_kws={"shrink": .5}
)

# gca -> get current axis
plt.gca().xaxis.tick_top()
plt.xticks(rotation=45, ha="left", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
# plt.tick_params(top=True, left=True, bottom=False, right=False)
plt.title("Correlation: Audio Metrics vs WER Degradation - SMALL MODEL")
plt.tight_layout()
plt.show()


Inpsecting correlation with WER

- CER must be removed, would cause leakage (0.97 correlation)
- RMS lin and DBFS global are 100% correlated. we will keep DBFS
- all other features have a low correlation to WER
- MFCCs standard deviation present a lot of collineraity within themselves. Will clean out anything above 0.6


In [ ]:
# Clean out the correlated features
corr_matrix = corr_matrix.abs()

# Isolate upper triangle of the matrix to avoid evaluating the same pair twice
# Using the upper triangle and dropping col if any values in the upper triangle col is above threshold will only remove features (col) from the right
# In this case here, since MFCC are ordered in terms of columns, this automatically keep the lower order ones.
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(np.bool))

collinearity_threshold = 0.60
redundant_features = [col for col in upper_triangle.columns if any(upper_triangle[col] > collinearity_threshold)]

# Restrict the dropping logic to MFCC standard deviation
mfcc_std_to_drop = [col for col in redundant_features if 'mfcc' in col and 'std' in col]
# pprint(redundant_features)
# pprint(mfcc_std_to_drop)

to_drop_set = set(mfcc_std_to_drop)
to_drop_set.add('cer')
to_drop_set.add('mtr.rms_lin_global')
# to_drop_set

pruned_features = [item for item in cols_to_correlate if item not in to_drop_set]

pprint(pruned_features)

In [ ]:
# ==  scatter plot of delta_wer and delta_zcr ==
fig, axes = plt.subplots(2, 2, figsize=(14, 6))

# Plot 1: Noisy ZCR vs WER
sns.scatterplot(x='mtr.mean_zcr', y='wer', data=df_smallModel, ax=axes[0,0], alpha=0.6)
axes[0,0].set_title('Noisy Mean ZCR vs WER Degradation - Small Model')
axes[0,0].set_xlabel("Noisy Audio ZCR")
axes[0,0].set_ylabel("WER")

# Plot 2: Original RMS vs WER
sns.scatterplot(x='mtr.rms_dbfs_global', y='wer', data=df_smallModel, ax=axes[0,1], alpha=0.6)
axes[0,1].set_title('Original Clean RMS vs WER Degradation - Small Model')
axes[0,1].set_xlabel('Clean RMS (dBFS)')
axes[0,1].set_ylabel('WER')

# Plot 1: Noisy ZCR vs WER
sns.scatterplot(x='mtr.mean_zcr', y='wer', data=df_largeModel, ax=axes[1,0], alpha=0.6)
axes[1,0].set_title('Noisy Mean ZCR vs WER Degradation - Large Model')
axes[1,0].set_xlabel("Noisy Audio ZCR")
axes[1,0].set_ylabel("WER")

# Plot 2: Original RMS vs WER
sns.scatterplot(x='mtr.rms_dbfs_global', y='wer', data=df_largeModel, ax=axes[1,1], alpha=0.6)
axes[1,1].set_title('Original Clean RMS vs WER Degradation - Large Model')
axes[1,1].set_xlabel('Clean RMS (dBFS)')
axes[1,1].set_ylabel('WER')

plt.tight_layout()
plt.show()

In [ ]:
plot_feature_scatter_grid(
    df_smallModel,
    pruned_features,  # 'wer' is filtered out automatically
    target="wer",
    ncols=4,
    title="WER vs pruned features - Small Model",
    # save_path="./images/wer_scatter_small.png",
)


In [ ]:
plot_feature_scatter_grid(
    df_largeModel,
    pruned_features,
    target="wer",
    ncols=4,
    title="WER vs pruned features - Large Model",
)

In [ ]:
# Plot Noist ZCR vs Delta WER with colored dots for RMS dBFS
sns.scatterplot(
    x='mtr.mean_zcr',
    y='wer',
    hue='mtr.min_zcr',
    palette='viridis',
    data=df_smallModel,
    alpha=0.8,
    s=60
)

plt.title('Noisy ZCR vs WER Degradation (Colored by mtr.min_zcr)')
plt.xlabel('Noisy Audio ZCR')
plt.ylabel('WER (Degradation)')

# Move the legend outside the plot
plt.legend(title='min ZCR', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Framework Speed Comparison (MLX vs MPS)

In [ ]:
# IS_APPLE_SILICON = platform.system() == "Darwin" and platform.processor() == "arm"
# IS_APPLE_SILICON= False
# transcriber = load_transcriber_engine()

def run_framework_speed_comparison(n_samples=10):
    # 1.load both engines
    force_mps = config.FORCE_MPS or True
    mlx_transcriber = load_transcriber_engine(IS_APPLE_SILICON=True)
    hf_transcriber = load_transcriber_engine(IS_APPLE_SILICON=False, FORCE_MPS=force_mps)

    # 2.Pre-load all samples to prevent biasing results from network latencies
    iterator = iter(dataset)
    samples = [next(iterator) for _ in range(n_samples + 1)]
    warmup_audio = samples[0]["audio"]["array"]
    test_samples = samples[1:]

    print(f"Warming up engines...")
    _ = mlx_transcriber(warmup_audio)
    _ = hf_transcriber(warmup_audio)

    # Pre-load ground truth text refs
    refs = []
    for s in test_samples:
        refs.append(cleanup_pipeline(s["text"]))

    # 3. Time engine A (MLX)
    print(f"Profiling Native MLX...")
    mlx_hyps = []
    mlx_start = time.perf_counter()
    for s in test_samples:
        mlx_hyps.append(cleanup_pipeline(mlx_transcriber(s["audio"]["array"])["text"]))
    mlx_time = time.perf_counter() - mlx_start

    # 4. Time engine 4 (PyTorch MPS)
    print(f"Profiling PyTorch {'mps' if force_mps else 'cpu'} pipeline...")
    hf_hyps =  []
    hf_start = time.perf_counter()
    for s in test_samples:
        hf_hyps.append(cleanup_pipeline(hf_transcriber(s["audio"]["array"])["text"]))
    hf_time = time.perf_counter() - hf_start

    # 5. Compute Metrics
    mlx_wer = jiwer.wer(refs, mlx_hyps)
    hf_wer = jiwer.wer(refs, hf_hyps)

    print(f"\nMLX Total Time   : {mlx_time:.2f}s | WER: {mlx_wer*100:.2f}%")
    print(f"PyTorch {'mps' if force_mps else 'cpu'} Time : {hf_time:.2f}s | WER: {hf_wer*100:.2f}%")
    print(f"MLX Speed Factor : {hf_time / mlx_time:.2f}x faster")

    return

In [ ]:
# == RUN FRAMEWORK COMPARISON TEST ==
if config.RUN_FRAMEWORK_SPEED_COMPARISON_TEST:
    run_framework_speed_comparison(n_samples=10)